In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print("All Libraries imported sucessfully")
print(f"pandas version :{pd.__version__}")
print(f"sqlite3 version :{sqlite3.version}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/student_performance.csv')

In [ ]:
print(df.head(5))

In [ ]:
conn=sqlite3.connect('college.db')
cursor=conn.cursor()
df.to_sql(
    'students',
    conn,
    if_exists='replace',
    index=False
)
cursor.execute("SELECT COUNT(*) FROM students")
count=cursor.fetchone()[0]
print(f"Database 'college_db' created successfully!")
print(f" Table 'students'has {count} rows")

In [ ]:
cursor.execute("PRAGMA table_info(students)")
columns_info=cursor.fetchall()
print("Table structure of 'students':")
print(f"{'Column Nane':<25} {'Data type':<12}")
print("-"*38)
for col in columns_info:
  print(f"{col[1]:<25} {col[2]:<12}")


In [ ]:
def run_query(sql,description=""):
  if description:
    print(f"\n{'='*55}")
    print(f"{description}")
    print(f"{'='*55}")
    result=pd.read_sql_query(sql,conn)
    print(result.to_string(index=False))
    return result
print("Helper function 'run_query'defined successfully" )
print("usage:run_query(sql_string,description_label)")



In [ ]:
query1 = """
SELECT student_id,name,department,math_score,attendance_percentage
FROM Students
LIMIT 5 OFFSET 25
"""
result1 = run_query(query1,"Query 1:Last 5 students(SELECT+LIMIT)")

In [ ]:
query2="""
SELECT name,department,math_score
FROM students
ORDER BY math_score DESC
LIMIT 5
"""
result2=run_query(query2,"Query 2:Top 5 students(ORDER BY+LIMIT)")

In [ ]:
query3="""
SELECT name,department,programming_score
FROM students
ORDER BY programming_score DESC
LIMIT 5
"""
result3=run_query(query3,"Query 3:Top 5 students(ORDER BY+LIMIT)")


In [ ]:
query4="""
SELECT name,department,programming_score
FROM students
WHERE programming_score BETWEEN 50 AND 75
ORDER BY programming_score DESC
"""
result4=run_query(query4,"Query 4:Students with programming score between 50 and 75")

In [ ]:
query5="""
SELECT name,math_score,science_score,programming_score,attendance_percentage
FROM students
WHERE department="Computer Science"
ORDER BY programming_score DESC
"""
result5=run_query(query5,"Query 5")

In [ ]:
query_department = """
SELECT DISTINCT department FROM students
"""
result_department = run_query(query_department, "Query: All Departments")

In [ ]:
query_overall_top_scorer = """
SELECT name,department, programming_score FROM students
WHERE department IN ('Computer Science', 'Electronics', 'Civil')
ORDER BY programming_score DESC
LIMIT 1;
"""
result_overall_top_scorer = run_query(query_overall_top_scorer, "Query: Overall Top Scorer in Programming from Selected Departments")

In [ ]:
query4="""
SELECT name,department,attendance_percentage
FROM students
WHERE attendance_percentage>90
AND department <> 'Civil'
ORDER BY attendance_percentage DESC
"""
result4=run_query(query4,"Query 4: Students with attendance percentage greater than 90 and not in Civil department")

In [ ]:
dept_data={
    'dept_code':['CS','EC','ME','CE'],
    'dept_name':['Computer Science','Electronics','Mechanical','Civil'],
    'hod_name':['Dr.Sharma','Dr.Reddy','Dr.Patel','Dr.Kumar'],
    'intake':[60,78,69,80]
}
dept_df=pd.DataFrame(dept_data)
print("Created 'departments'table ")
print(dept_df.to_string(index=False))
dept_map={
    'Conputer Science':'CS',
    'Electronics':'EC',
    'Mechanical':'ME',
    'Civil':'CE'
}
df['dept_code']=df['department'].map(dept_map)
df.to_sql('students',conn,if_exists='replace',index=False)
print("\nUpdated Students table with dept_code coloumn")

In [ ]:
query_join="""
select s.name,
s.math_score,
d.dept_name,
d.intake,
d.hod_name
from students AS s
inner join departments as d
on s.dept_code=d.dept_code
order by s.math_score DESC
limit 8
"""
result1=run_query(query_join,"query 13:(select+innerjoin+orderby)")

In [ ]:
chart1_sql = """
SELECT department,
       ROUND(AVG(math_score),2) AS avg_math
FROM students
GROUP BY department
ORDER BY avg_math DESC
"""
chart1_data = pd.read_sql_query(chart1_sql, conn)
plt.figure(figsize=(6,5))

plt.bar(chart1_data['department'],
        chart1_data['avg_math'])
plt.title("Average Math Score by Department")
plt.xlabel("Department")
plt.ylabel("Average Math Score")
plt.tight_layout()
plt.show()

In [ ]:
query = """
SELECT gender,
       MAX(programming_score) AS highest_programming_score
FROM students
GROUP BY gender
ORDER BY highest_programming_score DESC
"""
chart_data = pd.read_sql_query(query, conn)

plt.figure(figsize=(6,4))

plt.bar(chart_data['gender'],
        chart_data['highest_programming_score'])
plt.title("Highest Programming Score by Gender")
plt.xlabel("Gender")
plt.ylabel("Highest Programming Score")
plt.tight_layout()
plt.show()